# Dataset Inventory

Use this notebook to audit the local `.h5ad` files you have placed in `data/raw/`, inspect the dataset catalog extracted from the Word document, and choose the dataset you want to analyze next.


In [ ]:
from pathlib import Path
import sys

from IPython.display import display

CANDIDATES = [Path.cwd().resolve(), Path.cwd().resolve().parent]
REPO_ROOT = next((path for path in CANDIDATES if (path / "src").exists() and (path / "config").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Could not locate the repository root. Start Jupyter from the repo root or notebooks directory.")

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

for module_name in [name for name in list(sys.modules) if name == "cx47_oligo" or name.startswith("cx47_oligo.")]:
    del sys.modules[module_name]

from cx47_oligo.gene_panels import GENE_PANELS
from cx47_oligo.exports import ensure_results_dir, save_current_figure, save_json, save_table, save_text, slugify
from cx47_oligo.h5ad_tools import (
    adata_overview,
    dataset_catalog,
    discover_h5ad_files,
    expression_frame,
    load_h5ad,
    mean_scores_by_group,
    obs_column_summary,
    question_panel_table,
    suggest_obs_columns,
)
from cx47_oligo.notebook_viz import (
    composition_table,
    configure_notebook_style,
    plot_composition,
    plot_feature_heatmap,
    plot_gene_contribution,
    plot_group_counts,
    plot_scatter_relationship,
    plot_score_distributions,
    plot_score_heatmap,
    plot_top_ranked_genes,
    title_card_html,
)
from cx47_oligo.questions import QUESTION_BANK
from cx47_oligo.scanpy_tools import (
    available_question_marker_genes,
    assign_keyword_families,
    ensure_scanpy_adata,
    grouped_gene_expression,
    grouped_gene_expression_stats,
    obs_projection_frame,
    obs_keyword_mask,
    rank_genes_groups_df,
    score_question_panels_scanpy,
)

sc = configure_notebook_style(REPO_ROOT)


In [ ]:
display(
    title_card_html(
        "Dataset Inventory",
        "Audit the imported h5ad objects, inspect their metadata structure, and choose the strongest starting point for each biological question.",
        dataset_hint="Current default datasets: Jäkel for MS lesion-state questions, Falcão for EAE stress and reactive-glia questions.",
    )
)


In [ ]:
catalog = dataset_catalog()
display(catalog)


In [ ]:
DATA_ROOT = REPO_ROOT / "data" / "raw"
LOCAL_H5AD_FILES = discover_h5ad_files(DATA_ROOT)
LOCAL_H5AD_FILES


In [ ]:
DATASET_PATH = LOCAL_H5AD_FILES[0] if LOCAL_H5AD_FILES else None
DATASET_PATH


In [ ]:
RESULTS_DIR = ensure_results_dir(REPO_ROOT, "00_dataset_inventory", DATASET_PATH)
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sc.settings.figdir = FIGURES_DIR
print("Results will be written to:", RESULTS_DIR)
save_table(catalog, RESULTS_DIR / "dataset_catalog_snapshot.csv", index=False)
save_json({"dataset_path": str(DATASET_PATH) if DATASET_PATH else "", "results_dir": str(RESULTS_DIR)}, RESULTS_DIR / "run_metadata.json")


In [ ]:
adata = load_h5ad(DATASET_PATH) if DATASET_PATH else None
if adata is None:
    print("No .h5ad file found under data/raw yet.")
else:
    overview = adata_overview(adata)
    suggestions = suggest_obs_columns(adata)
    obs_summary = obs_column_summary(adata)
    display(overview)
    display(suggestions)
    display(obs_summary.head(25))
    save_table(overview, RESULTS_DIR / "adata_overview.csv", index=False)
    save_table(suggestions, RESULTS_DIR / "suggested_obs_columns.csv", index=False)
    save_table(obs_summary, RESULTS_DIR / "obs_column_summary.csv", index=False)
    print("scanpy version:", sc.__version__)


In [ ]:
DEFAULT_GROUP = next(
    (column for column in ["Celltypes", "Renamed_clusternames", "Lesion", "Group", "Condition"] if adata is not None and column in adata.obs.columns),
    None,
)

if adata is not None and DEFAULT_GROUP is not None:
    group_counts = plot_group_counts(adata.obs, DEFAULT_GROUP)
    display(group_counts)
    save_table(group_counts, RESULTS_DIR / f"cell_counts_by_{slugify(DEFAULT_GROUP)}.csv")
    save_current_figure(FIGURES_DIR / f"cell_counts_by_{slugify(DEFAULT_GROUP)}.png")


## Next Step

Once you identify a useful dataset and the relevant `.obs` columns for timepoint, condition, and cell state, move to one of the question-specific notebooks in this folder.
